# 📊 Finance RAG Chatbot using LangChain, FAISS & Hugging Face

## Project Overview

This project demonstrates how to build a **Retrieval-Augmented Generation (RAG)** chatbot capable of answering questions from financial documents such as annual reports (10-Ks), earnings reports, and investment guides.

Instead of relying only on an LLM's pre-trained knowledge, the chatbot retrieves the most relevant information from uploaded financial PDFs and generates accurate, context-aware responses.

### Objectives
- Build a Finance AI Assistant using RAG
- Process multiple financial PDF documents
- Generate vector embeddings using Sentence Transformers
- Store embeddings in a FAISS vector database
- Retrieve relevant document chunks using semantic search
- Generate responses using a Large Language Model (LLM)

### Technologies Used
- Python
- Jupyter Notebook
- LangChain
- FAISS
- Sentence Transformers
- Hugging Face Transformers
- PyPDF
- NumPy
- Pandas

### Workflow

Financial PDFs
→ PDF Loader
→ Text Chunking
→ Embedding Generation
→ FAISS Vector Database
→ Semantic Retrieval
→ Large Language Model
→ Finance Chatbot Response

### Dataset
The chatbot uses publicly available annual reports from companies such as:
- Apple
- Google
- Amazon
- NVIDIA
- Tesla

### Expected Output
Users can ask questions like:
- What was Apple's revenue in 2024?
- Compare Google and Apple Annual Stipend?
- What risks did Tesla mention in its annual report?
- Which company spent the most on Research & Development?

---

**Author:** Nupur  
**Project:** Finance RAG Chatbot  
**Version:** 1.0

### Install Libraries

In [17]:
!pip install langchain
!pip install langchain-community
!pip install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install transformers
!pip install torch

  Using cached faiss_cpu-1.14.3-cp313-cp313-win_amd64.whl.metadata (7.8 kB)
Using cached faiss_cpu-1.14.3-cp313-cp313-win_amd64.whl (16.2 MB)
  Using cached pypdf-6.14.2-py3-none-any.whl.metadata (7.2 kB)
Using cached pypdf-6.14.2-py3-none-any.whl (349 kB)


In [3]:
!pip install -U langchain-text-splitters

In [13]:
pip install langchain


Note: you may need to restart the kernel to use updated packages.


In [17]:
pip install langchain langchain-community


In [8]:
!pip install langchain-pdf

In [13]:
!pip install -U langchain-text-splitters
!pip install -U langchain-huggingface
!pip install -U langchain-community

### Import Libraries

In [14]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [4]:
import os

from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings

In [15]:
import langchain
from langchain_community.document_loaders import PyPDFLoader


In [18]:
import langchain_community
print(langchain_community.__version__)


0.4.2


In [12]:
import langchain

print(langchain.__version__)

1.3.14


In [15]:
import langchain_text_splitters

print("Installed successfully!")

Installed successfully!


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

### Load Every PDF

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

folder = r"C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data"

for file in os.listdir(folder):

    if file.endswith(".pdf"):

        loader = PyPDFLoader(os.path.join(folder, file))

        documents.extend(loader.load())

print(f"Loaded {len(documents)} pages.")

C:\Users\nupur\AppData\Local\Temp\ipykernel_233028\3529437204.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 521 pages.


In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader

folder = r"C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data"

print("Folder exists:", os.path.exists(folder))
print("Files:", os.listdir(folder))

Folder exists: True
Files: ['.ipynb_checkpoints', 'Amazon_2024.pdf', 'Apple_2024.pdf', 'Google_2024.pdf', 'NVDIA_2024.pdf', 'Tesla_2024.pdf']


In [10]:
import os
from langchain_community.document_loaders import PyPDFLoader

documents = []

folder = r"C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data"

for file in os.listdir(folder):

    if file.lower().endswith(".pdf"):

        print(f"Loading {file}...")

        loader = PyPDFLoader(os.path.join(folder, file))

        docs = loader.load()

        print(f"  Pages: {len(docs)}")

        documents.extend(docs)

print("=" * 50)
print(f"Total pages loaded: {len(documents)}")

Loading Amazon_2024.pdf...
  Pages: 91
Loading Apple_2024.pdf...
  Pages: 121
Loading Google_2024.pdf...
  Pages: 99
Loading NVDIA_2024.pdf...
  Pages: 174
Loading Tesla_2024.pdf...
  Pages: 36
Total pages loaded: 521


# Step 2: Text Chunking

## Objective

Large Language Models (LLMs) have a limited context window and cannot process entire PDF documents efficiently.

To improve retrieval performance, the loaded documents are divided into smaller overlapping chunks. These chunks preserve contextual information while enabling efficient semantic search.

### Why Chunking?

- Reduces context size
- Improves retrieval accuracy
- Preserves surrounding information using overlap
- Generates better embeddings

### Parameters

- Chunk Size: 500 characters
- Chunk Overlap: 100 characters

The resulting chunks will be stored in a vector database in the next step.

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 4431


# Step 3: Create Embeddings

## Objective

Text cannot be searched semantically in its raw form. Therefore, each text chunk is converted into a numerical vector called an **embedding**.

Embeddings capture the semantic meaning of the text, allowing the chatbot to retrieve relevant information even when the user's question does not exactly match the document wording.

### Embedding Model

**sentence-transformers/all-MiniLM-L6-v2**

- Fast and lightweight
- Produces 384-dimensional embeddings
- Excellent for semantic search and RAG applications

### Output

Each document chunk is converted into a vector representation that will later be stored in a FAISS vector database.

In [19]:
!pip install -U langchain-huggingface sentence-transformers

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings

In [21]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


# Step 4: Build the FAISS Vector Database

## Objective

After generating embeddings, they are stored in a **FAISS (Facebook AI Similarity Search)** vector database.

FAISS enables fast semantic similarity search over thousands of document chunks.

### Why FAISS?

- High-performance vector search
- Scales to millions of embeddings
- Optimized for Retrieval-Augmented Generation (RAG)

In [22]:
!pip install -U faiss-cpu

In [23]:
from langchain_community.vectorstores import FAISS

In [ ]:
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("FAISS Vector Database Created Successfully!")

In [ ]:
print(embedding_model)

In [26]:
print("Hello")

Hello


In [27]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Model loaded")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded


In [28]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded successfully!


In [29]:
import torch
print(torch.__version__)

2.13.0+cpu


In [30]:
import sentence_transformers
print(sentence_transformers.__version__)

5.6.1


In [31]:
test_chunks = chunks[:10]

test_db = FAISS.from_documents(
    documents=test_chunks,
    embedding=embedding_model
)

print("✅ FAISS works!")

✅ FAISS works!


In [32]:
test_chunks = chunks[:100]

test_db = FAISS.from_documents(
    documents=test_chunks,
    embedding=embedding_model
)

print("✅ 100 chunks completed!")

✅ 100 chunks completed!


In [33]:
from langchain_community.vectorstores import FAISS

batch_size = 500

vector_db = None

for i in range(0, len(chunks), batch_size):

    batch = chunks[i:i + batch_size]

    if vector_db is None:
        vector_db = FAISS.from_documents(
            documents=batch,
            embedding=embedding_model
        )
    else:
        vector_db.add_documents(batch)

    print(f"Processed {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")

print("✅ FAISS database created successfully!")

Processed 500/4431 chunks
Processed 1000/4431 chunks
Processed 1500/4431 chunks
Processed 2000/4431 chunks
Processed 2500/4431 chunks
Processed 3000/4431 chunks
Processed 3500/4431 chunks
Processed 4000/4431 chunks
Processed 4431/4431 chunks
✅ FAISS database created successfully!


In [34]:
vector_db.save_local("vector_db")

print("Vector Database Saved!")

Vector Database Saved!


In [35]:
query = "What was Apple's total revenue?"

results = vector_db.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("=" * 80)
    print(doc.page_content[:500])
    print("\nSource:", doc.metadata)


Result 1
Apple Inc.
CONSOLIDATED STATEMENTS OF OPERATIONS(In millions, except number of shares, which are reflected in thousands, and per-share amounts)
Years ended
September 28,2024 September 30,2023 September 24,2022
Net sales:
   Products $ 294,866 $ 298,085 $ 316,199 
   Services 96,169 85,200 78,129 
Total net sales 391,035 383,285 394,328 
Cost of sales:
   Products 185,233 189,282 201,471 
   Services 25,119 24,855 22,075 
Total cost of sales 210,352 214,137 223,546

Source: {'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2024-11-01T06:05:37-04:00', 'title': '0000320193-24-000123', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2024-11-01 for the period ending 2024-09-28', 'keywords': '0000320193-24-000123; ; 10-K', 'moddate': '2024-11-01T06:06:09-04:00', 'source': 'C:\\Users\\nupur\\Projects\\Finance-RAG-Chatbot\\Data\\Apple_2024.pdf', 'total_pages': 121, 'page': 31

# 🤖 Step 5: Integrate a Large Language Model (LLM)

## Objective

The FAISS vector database can retrieve the most relevant document chunks, but it cannot generate natural language responses. To transform the system into a Retrieval-Augmented Generation (RAG) chatbot, an LLM is integrated.

The LLM receives the retrieved financial context and generates an accurate, human-readable answer while remaining grounded in the retrieved documents.

---

## Why Use an LLM?

A vector database performs **semantic retrieval**, whereas an LLM performs **natural language generation**.

The combined workflow enables:

- Context-aware question answering
- Financial report summarization
- Company performance analysis
- Comparative analysis across multiple reports
- Natural language responses grounded in retrieved documents

---

## LLM Used

For this project, **Llama 3.1 8B Instant** served through the **Groq API** is recommended.

### Advantages

- Very fast inference
- High-quality responses
- Free developer API
- Suitable for Retrieval-Augmented Generation (RAG)
- Easy integration with Python

---

## RAG Workflow

```text
User Question
       │
       ▼
FAISS Similarity Search
       │
       ▼
Top-k Relevant Financial Chunks
       │
       ▼
Prompt Construction
       │
       ▼
Llama 3.1 Large Language Model
       │
       ▼
Context-Aware Financial Answer
```

---

## Prompt Engineering Strategy

The prompt instructs the model to:

- Answer only from the retrieved financial reports.
- Avoid generating unsupported information.
- Respond professionally and accurately.
- Inform the user when the answer is unavailable in the provided documents.

This significantly reduces hallucinations and improves response reliability.

---

## Expected Output

**Question**

> What was Apple's total revenue in 2024?

**Answer**

Apple reported total net sales of approximately \$391 billion during fiscal year 2024 according to its Annual Report.

**Source**

Apple_2024.pdf

---

## Learning Outcome

After completing this step, the Finance RAG Chatbot will be able to:

- Retrieve relevant financial information using semantic search.
- Generate context-aware responses using an LLM.
- Answer questions across multiple company annual reports.
- Demonstrate a complete Retrieval-Augmented Generation (RAG) pipeline suitable for real-world financial document analysis.

In [36]:
!pip install groq

In [ ]:
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

In [37]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [38]:
def ask_finance_bot(question):
    # Retrieve relevant documents
    docs = vector_db.similarity_search(question, k=3)

    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a professional financial analyst.

Answer ONLY using the information provided in the context below.

If the answer is not present in the documents, say:
"I could not find that information in the provided financial reports."

Context:
{context}

Question:
{question}

Answer:
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content

In [39]:
question = "What was Apple's total revenue in 2024?"

answer = ask_finance_bot(question)

print(answer)

I could not find that information in the provided financial reports.


In [40]:
query = "What was Apple's total revenue in 2024?"

docs = vector_db.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("=" * 80)
    print(f"Result {i+1}")
    print(doc.metadata)
    print(doc.page_content[:1000])

Result 1
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2024-11-01T06:05:37-04:00', 'title': '0000320193-24-000123', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2024-11-01 for the period ending 2024-09-28', 'keywords': '0000320193-24-000123; ; 10-K', 'moddate': '2024-11-01T06:06:09-04:00', 'source': 'C:\\Users\\nupur\\Projects\\Finance-RAG-Chatbot\\Data\\Apple_2024.pdf', 'total_pages': 121, 'page': 37, 'page_label': '38'}
Total net sales include $7.7 billion of revenue recognized in 2024 that was included in deferred revenue as of September 30, 2023, $8.2 billion of revenuerecognized in 2023 that was included in deferred revenue as of September 24, 2022, and $7.5 billion of revenue recognized in 2022 that was included indeferred revenue as of September 25, 2021.
® ®
(1)
Apple Inc. | 2024 Form 10-K | 35
Result 2
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator

In [41]:
query = "What was Apple's total revenue in 2024?"

docs = vector_db.similarity_search(query, k=5)

for i, doc in enumerate(docs):
    print("=" * 80)
    print(doc.metadata)
    print(doc.page_content)

{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2024-11-01T06:05:37-04:00', 'title': '0000320193-24-000123', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2024-11-01 for the period ending 2024-09-28', 'keywords': '0000320193-24-000123; ; 10-K', 'moddate': '2024-11-01T06:06:09-04:00', 'source': 'C:\\Users\\nupur\\Projects\\Finance-RAG-Chatbot\\Data\\Apple_2024.pdf', 'total_pages': 121, 'page': 37, 'page_label': '38'}
Total net sales include $7.7 billion of revenue recognized in 2024 that was included in deferred revenue as of September 30, 2023, $8.2 billion of revenuerecognized in 2023 that was included in deferred revenue as of September 24, 2022, and $7.5 billion of revenue recognized in 2022 that was included indeferred revenue as of September 25, 2021.
® ®
(1)
Apple Inc. | 2024 Form 10-K | 35
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing H

In [42]:
def ask_finance_bot(question):
    # Retrieve relevant chunks
    docs = vector_db.similarity_search(question, k=5)

    # Combine retrieved context
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are an expert financial analyst.

Use ONLY the information provided in the context below.

If the answer exists, provide a clear and concise response.

If the answer is partially available, summarize the available information.

Only respond with:
"I could not find that information in the provided financial reports."
if the information is completely absent.

==========================
CONTEXT
==========================

{context}

==========================
QUESTION
==========================

{question}

==========================
ANSWER
==========================
"""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You answer questions only using the supplied financial documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
    )

    return response.choices[0].message.content

In [43]:
answer = ask_finance_bot("What was Apple's total revenue in 2024?")
print(answer)

Total net sales in 2024 was $391,035 million.


In [44]:
def ask_finance_bot(question):
    docs = vector_db.similarity_search(question, k=5)

    context = "\n\n".join(doc.page_content for doc in docs)

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "You are a professional financial analyst. Answer only using the provided context."
            },
            {
                "role": "user",
                "content": f"""
Context:
{context}

Question:
{question}

Answer:
"""
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    sources = list(set(doc.metadata["source"] for doc in docs))

    print("\nAnswer:\n")
    print(answer)

    print("\nSources:")
    for source in sources:
        print("-", source)

In [45]:
ask_finance_bot("What was Apple's total revenue in 2024?")


Answer:

According to the provided context, Apple's total net sales in 2024 was $391,035 million.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf


In [46]:
ask_finance_bot("What were Apple's operating expenses?")


Answer:

To determine Apple's operating expenses, we need to look at the total cost of sales and subtract it from the total net sales, then add back the depreciation and amortization, and the share-based compensation expense.

Total net sales: $391,035 million (2024), $383,285 million (2023), $394,328 million (2022)

Total cost of sales: $210,352 million (2024), $214,137 million (2023), $223,546 million (2022)

Operating expenses (before adjustments) = Total cost of sales
2024: $210,352 million
2023: $214,137 million
2022: $223,546 million

Now, let's add back the depreciation and amortization, and the share-based compensation expense to get the operating expenses.

Operating expenses (2024) = $210,352 million + $11,445 million + $11,688 million = $233,485 million

Operating expenses (2023) = $214,137 million + $11,519 million + $10,833 million = $236,489 million

Operating expenses (2022) = $223,546 million + $11,104 million + $9,038 million = $243,688 million

However, we also need 

In [47]:
ask_finance_bot("How much did Apple spend on research and development?")


Answer:

Unfortunately, the provided context does not directly mention the total amount spent on research and development. However, it does provide the percentage of total net sales for research and development for the years 2024, 2025, 2026, 2027, 2028, and 2029.

The percentages are:
- 2025: 15%
- 2026: 14%
- 2027: 13%

However, we do not have the total net sales for 2024. We do have the total net sales for the years 2025-2029, which are $3,206 million, $2,440 million, $1,156 million, $3,121 million, and $633 million, respectively.

To calculate the total net sales for 2024, we need to know the total net sales for the years 2024-2029. Unfortunately, the provided context does not provide this information.

However, we can calculate the total net sales for the years 2025-2029, which is $3,206 + $2,440 + $1,156 + $3,121 + $633 = $10,756 million.

We also know that the total net sales for the years 2025-2029 is $10,756 million, and the percentages of total net sales for research and dev

In [48]:
ask_finance_bot("What was Amazon's total revenue?")


Answer:

$638B.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf


In [49]:
ask_finance_bot("What are Amazon's major business segments?")


Answer:

Amazon's major business segments are:

1. North America
2. International
3. Amazon Web Services (AWS)

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf


In [50]:
ask_finance_bot("How much advertising revenue did Google generate?")


Answer:

To find the total advertising revenue generated by Google, we need to add the revenues from Google Search & other, YouTube ads, and Google Network properties.

From the table, we can see that:

- Google Search & other revenue in 2024 was $198,084 million.
- YouTube ads revenue in 2024 was $36,147 million.
- Google Network revenue in 2024 was $30,359 million.

Adding these together, we get:

$198,084 million + $36,147 million + $30,359 million = $264,590 million

However, we also need to consider that Google advertising revenues are comprised of these three components. The table also shows that Google advertising revenues in 2024 were $264,590 million.

Therefore, the total advertising revenue generated by Google in 2024 was $264,590 million.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Google_2024.pdf


In [51]:
ask_finance_bot("What were Tesla's automotive revenues?")


Answer:

Tesla's automotive revenues for the given periods are as follows:

- Q4-2023: $21,563 million
- Q1-2024: $17,378 million
- Q2-2024: $19,878 million
- Q3-2024: $20,016 million
- Q4-2024: $19,798 million

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf


In [52]:
ask_finance_bot("What were Tesla's automotive revenues?")


Answer:

Tesla's automotive revenues for the given periods are as follows:

- Q4-2023: $21,563 million
- Q1-2024: $17,378 million
- Q2-2024: $19,878 million
- Q3-2024: $20,016 million
- Q4-2024: $19,798 million

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf


In [53]:
ask_finance_bot("What were NVIDIA's AI business highlights?")


Answer:

NVIDIA has already applied AI to build several multi-billion-dollar verticals, including:

1. Gaming
2. Healthcare
3. Automotive
4. Robotics

Additionally, NVIDIA is now bringing its accelerated computing and AI to the enterprise, expanding the market for its workstation-class GPUs as more enterprise customers develop and deploy AI applications with their data on-premises.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\NVDIA_2024.pdf


In [54]:
ask_finance_bot("Compare Apple and Amazon revenues.")


Answer:

Based on the provided context, we can compare Apple and Amazon revenues as follows:

- Apple's total net sales in 2024 were $391,035 million.
- Amazon's total net sales in 2024 were $637,959 million.

Amazon's revenue in 2024 was approximately 63.5% higher than Apple's revenue in the same year.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf


In [55]:
ask_finance_bot("Which company reported the highest net sales?")


Answer:

Based on the provided information, the company reported the highest net sales is the one with the consolidated net sales of $637,959 in 2024.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf


In [56]:
ask_finance_bot("Which company invested the most in research and development?")


Answer:

Based on the provided information, it is not explicitly stated which company is being referred to. However, we can analyze the data provided to determine the company's research and development expenses.

The data shows that the company's research and development expenses were $45,427 in 2023 and $49,326 in 2024. This represents an increase of $3.9 billion from 2023 to 2024.

To determine which company invested the most in research and development, we need to compare the research and development expenses of the company in question with other companies. However, based on the provided data, we can conclude that the company in question invested $49,326 in research and development in 2024, which is the highest amount mentioned in the data.

Therefore, the company invested the most in research and development in 2024.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Google_2024.pdf
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf


# 🤖 Step 6: Interactive Finance RAG Chatbot

## Objective

The previous step demonstrated how to answer a single financial question using Retrieval-Augmented Generation (RAG).

In this step, an interactive chatbot interface is created that allows users to ask multiple financial questions without restarting the notebook.

The chatbot performs the following operations for each query:

1. Accepts a user's financial question.
2. Retrieves the most relevant document chunks using FAISS.
3. Sends the retrieved context to the Large Language Model.
4. Generates a context-aware response.
5. Displays the source document(s).

This simulates a real-world conversational AI assistant for financial document analysis.

In [57]:
while True:

    question = input("\n💬 Ask a financial question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\n👋 Thank you for using the Finance RAG Chatbot!")
        break

    ask_finance_bot(question)

    print("\n" + "="*80)


💬 Ask a financial question (type 'exit' to quit):  What's Google annual stipend?



Answer:

The information provided does not directly mention Google's annual stipend. However, it does mention Alphabet-level activities, which include costs not allocated to specific segments. 

In 2023, Alphabet-level activities were $9,186 million, and in 2024, they were $10,541 million.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Google_2024.pdf




💬 Ask a financial question (type 'exit' to quit):  hi



Answer:

It appears that you've provided a snippet from Alphabet Inc.'s (Google's parent company) proxy statement for the 2024 Meeting. I'll assume you're asking about the context of this document.

As a financial analyst, I can provide some context. This document is likely a proxy statement filed with the Securities and Exchange Commission (SEC) as part of Alphabet Inc.'s annual meeting process. The document contains information about the company's business, financial performance, and governance.

The numbers you provided (63, 43, and 33) might be related to the number of questions submitted by stockholders or the number of proposals presented at the meeting.

However, without more context or information about the specific questions you're asking, it's difficult to provide a more detailed answer. If you have a specific question about the proxy statement or Alphabet Inc.'s financial performance, I'd be happy to try and assist you.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbo


💬 Ask a financial question (type 'exit' to quit):  quit



Answer:

It seems like you're asking me to quit my analysis or my role as a financial analyst. However, based on the context provided, I'll assume you're asking about the concept of quitting in the context of learning and growth.

In that case, quitting learning is not an option for individuals or companies that want to stay competitive and innovative. As mentioned in the context, the day we stop learning is the day we risk undermining what we're capable of building in the future.

Sources:
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Google_2024.pdf
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf
- C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf




💬 Ask a financial question (type 'exit' to quit):  exit



👋 Thank you for using the Finance RAG Chatbot!


In [58]:
import pandas as pd

# -------------------------------
# Store Chat History
# -------------------------------

chat_history = []


# -------------------------------
# Finance RAG Chatbot Function
# -------------------------------

def ask_finance_bot(question):

    # Retrieve top 5 relevant chunks
    docs = vector_db.similarity_search(
        question,
        k=5
    )

    # Build context
    context = "\n\n".join(
        doc.page_content for doc in docs
    )

    # Prompt
    prompt = f"""
You are an expert financial analyst.

Answer ONLY using the information provided in the context.

If the answer exists,
provide a clear and concise response.

If the answer does not exist,
reply:

"I could not find that information in the provided financial reports."

-------------------------
CONTEXT
-------------------------

{context}

-------------------------
QUESTION
-------------------------

{question}

-------------------------
ANSWER
-------------------------
"""

    # Send prompt to Groq
    response = client.chat.completions.create(

        model="llama-3.1-8b-instant",

        messages=[
            {
                "role": "system",
                "content": "You answer financial questions using only the retrieved documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0

    )

    answer = response.choices[0].message.content

    # -------------------------------
    # Save Chat History
    # -------------------------------

    chat_history.append({
        "Question": question,
        "Answer": answer
    })

    # -------------------------------
    # Display Answer
    # -------------------------------

    print("\n" + "="*80)
    print("💬 ANSWER")
    print("="*80)

    print(answer)

    # -------------------------------
    # Display Sources
    # -------------------------------

    print("\n📚 SOURCES")

    shown = set()

    for doc in docs:

        source = doc.metadata.get("source", "Unknown Source")
        page = doc.metadata.get("page", "Unknown")

        key = (source, page)

        if key not in shown:
            print(f"• {source} (Page {page + 1})")
            shown.add(key)

    print("="*80)

    return answer

In [59]:
while True:

    question = input("\nAsk a financial question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("\nThank you for using the Finance RAG Chatbot!")
        break

    ask_finance_bot(question)


Ask a financial question (type 'exit' to quit):  Compare Apple and Amazon revenues.



💬 ANSWER
I could not find that information in the provided financial reports.

📚 SOURCES
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 2)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 26)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 32)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 49)



Ask a financial question (type 'exit' to quit):  Compare Apple and Amazon revenues.?



💬 ANSWER
I could not find that information in the provided financial reports.

📚 SOURCES
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 2)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 26)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 49)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Amazon_2024.pdf (Page 32)



Ask a financial question (type 'exit' to quit):  What was Apple's total revenue in 2024?



💬 ANSWER
$391,035 million

📚 SOURCES
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 38)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 48)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 27)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Apple_2024.pdf (Page 32)



Ask a financial question (type 'exit' to quit):  What were Tesla's automotive revenues?



💬 ANSWER
Total automotive revenues: 
Q4-2023: 21,563
Q1-2024: 17,378
Q2-2024: 19,878
Q3-2024: 20,016
Q4-2024: 19,798

📚 SOURCES
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf (Page 29)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf (Page 4)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf (Page 8)
• C:\Users\nupur\Projects\Finance-RAG-Chatbot\Data\Tesla_2024.pdf (Page 7)



Ask a financial question (type 'exit' to quit):  exit



Thank you for using the Finance RAG Chatbot!


In [60]:
print("\nCHAT HISTORY")
print("="*80)

for i, chat in enumerate(chat_history, start=1):

    print(f"\nQuestion {i}")

    print(chat["Question"])

    print("\nAnswer")

    print(chat["Answer"])

    print("-"*80)


CHAT HISTORY

Question 1
Compare Apple and Amazon revenues.

Answer
I could not find that information in the provided financial reports.
--------------------------------------------------------------------------------

Question 2
Compare Apple and Amazon revenues.?

Answer
I could not find that information in the provided financial reports.
--------------------------------------------------------------------------------

Question 3
What was Apple's total revenue in 2024?

Answer
$391,035 million
--------------------------------------------------------------------------------

Question 4
What were Tesla's automotive revenues?

Answer
Total automotive revenues: 
Q4-2023: 21,563
Q1-2024: 17,378
Q2-2024: 19,878
Q3-2024: 20,016
Q4-2024: 19,798
--------------------------------------------------------------------------------


In [61]:
df = pd.DataFrame(chat_history)

df.to_csv(
    "Finance_Chat_History.csv",
    index=False
)

print("Chat history exported successfully!")

Chat history exported successfully!


In [62]:
print(vector_db)

In [63]:
docs = vector_db.similarity_search("Apple revenue", k=3)

print(f"Retrieved {len(docs)} documents")

Retrieved 3 documents


In [64]:
for i, doc in enumerate(docs, 1):
    print(f"\nResult {i}")
    print(doc.metadata)
    print(doc.page_content[:500])


Result 1
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creator': 'EDGAR Filing HTML Converter', 'creationdate': '2024-11-01T06:05:37-04:00', 'title': '0000320193-24-000123', 'author': 'EDGAR® Online LLC, a subsidiary of OTC Markets Group', 'subject': 'Form 10-K filed on 2024-11-01 for the period ending 2024-09-28', 'keywords': '0000320193-24-000123; ; 10-K', 'moddate': '2024-11-01T06:06:09-04:00', 'source': 'C:\\Users\\nupur\\Projects\\Finance-RAG-Chatbot\\Data\\Apple_2024.pdf', 'total_pages': 121, 'page': 37, 'page_label': '38'}
Total net sales include $7.7 billion of revenue recognized in 2024 that was included in deferred revenue as of September 30, 2023, $8.2 billion of revenuerecognized in 2023 that was included in deferred revenue as of September 24, 2022, and $7.5 billion of revenue recognized in 2022 that was included indeferred revenue as of September 25, 2021.
® ®
(1)
Apple Inc. | 2024 Form 10-K | 35

Result 2
{'producer': 'EDGRpdf Service w/ EO.Pdf 22.0.40.0', 'creat

In [65]:
import pandas as pd

# Chat history
chat_history = []


def ask_finance_bot(question):

    # Retrieve documents
    docs = vector_db.similarity_search(question, k=5)

    # Create context
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""
You are a professional financial analyst.

Answer ONLY using the information provided below.

Context:
{context}

Question:
{question}

Answer:
"""

    # Ask Groq
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "system",
                "content": "Answer only from the retrieved financial reports."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    answer = response.choices[0].message.content

    # Save history
    chat_history.append({
        "Question": question,
        "Answer": answer
    })

    print("\n" + "="*80)
    print("ANSWER")
    print("="*80)
    print(answer)

    print("\nSOURCES")
    print("="*80)

    shown = set()

    for doc in docs:

        source = doc.metadata.get("source", "Unknown")

        page = doc.metadata.get("page", 0)

        filename = source.split("\\")[-1]

        if (filename, page) not in shown:

            print(f"{filename} | Page {page+1}")

            shown.add((filename, page))

    print("="*80)

    return answer

In [66]:
ask_finance_bot("What was Apple's total revenue in 2024?")


ANSWER
According to the provided information, Apple's total net sales in 2024 was $391,035 million.

SOURCES
Apple_2024.pdf | Page 38
Apple_2024.pdf | Page 48
Apple_2024.pdf | Page 27
Apple_2024.pdf | Page 32


"According to the provided information, Apple's total net sales in 2024 was $391,035 million."

In [67]:
while True:

    question = input("\nAsk a financial question (type 'exit' to quit): ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    ask_finance_bot(question)


Ask a financial question (type 'exit' to quit):  Who is CEO of Google?



ANSWER
Sundar Pichai, under the leadership of Alphabet.

SOURCES
Google_2024.pdf | Page 5
Amazon_2024.pdf | Page 18
Amazon_2024.pdf | Page 17



Ask a financial question (type 'exit' to quit):  Compare profit of NVDIA with Amazon?



ANSWER
I cannot provide information about NVIDIA's profit. Is there anything else I can help you with?

SOURCES
Amazon_2024.pdf | Page 2
Amazon_2024.pdf | Page 49
Amazon_2024.pdf | Page 32



Ask a financial question (type 'exit' to quit):  Annual earning of NVDIA?


...


ANSWER
Based on the provided information, here are some key points from the financial reports of NVIDIA:

1. **Fiscal Year Ended**: January 28, 2024
2. **Audited by**: Independent Registered Public Accounting Firm
3. **Internal Audit Function**: Reports to the Audit Committee (AC) and is responsible for reviewing and evaluating internal controls and business processes.
4. **Annual Internal Audit Plan**: Approved by the AC and monitored for activities and performance.
5. **Form 10-K**: Filed with the SEC for the fiscal year ended January 28, 2024, and furnished to shareholders concurrently.
6. **Additional Copies**: Available upon written request to Investor Relations, NVIDIA Corporation, 2788 San Tomas Expressway, Santa Clara, CA 95051 or shareholdermeeting@nvidia.com.

SOURCES
NVDIA_2024.pdf | Page 135
NVDIA_2024.pdf | Page 83
NVDIA_2024.pdf | Page 167
NVDIA_2024.pdf | Page 87
NVDIA_2024.pdf | Page 170



Ask a financial question (type 'exit' to quit):  quit



ANSWER
Based on the provided information, I couldn't find a direct answer to the question "quit". However, I can infer that the concept of quitting learning is mentioned in the context of Amazon, where it's stated that "the day we stop learning at Amazon is the day we risk undermining what we're capable of building in the future."

SOURCES
Amazon_2024.pdf | Page 4
Google_2024.pdf | Page 11
Amazon_2024.pdf | Page 75
Tesla_2024.pdf | Page 22
Tesla_2024.pdf | Page 28



Ask a financial question (type 'exit' to quit):  exit


Goodbye!


In [68]:
for i, chat in enumerate(chat_history, 1):

    print(f"\nQuestion {i}")
    print(chat["Question"])

    print("\nAnswer")
    print(chat["Answer"])


Question 1
What was Apple's total revenue in 2024?

Answer
According to the provided information, Apple's total net sales in 2024 was $391,035 million.

Question 2
Who is CEO of Google?

Answer
Sundar Pichai, under the leadership of Alphabet.

Question 3
Compare profit of NVDIA with Amazon?

Answer
I cannot provide information about NVIDIA's profit. Is there anything else I can help you with?

Question 4
Annual earning of NVDIA?

Answer
According to the Consolidated Statements of Comprehensive Income, NVIDIA's annual earnings for the year ended January 28, 2024, is $29,760 million.

Question 5
Best apple scheme?

Answer
Based on the provided information, I can analyze the upcoming product releases and operating system updates to make a recommendation.

The upcoming product releases and updates include:

- MacBook Air 13-in. and 15-in. (Second Quarter 2024)
- iPad Air and iPad Pro (Third Quarter 2024)
- iPhone 16 series, Apple Watch Series 10, and AirPods 4 (Fourth Quarter 2024)
- Upda

In [69]:
import pandas as pd

df = pd.DataFrame(chat_history)

df.to_csv("Finance_Chat_History.csv", index=False)

print("✅ Chat history exported successfully!")

✅ Chat history exported successfully!


# 🌐 Step 11: Build a Streamlit Web Application

## Objective

The Finance RAG Chatbot currently operates inside a Jupyter Notebook.

To create a production-ready AI application, the chatbot will be deployed as a **Streamlit Web Application**.

The web application will provide:

- Interactive chat interface
- Real-time financial question answering
- Source document references
- Chat history
- Clean and responsive UI
- Easy deployment on Streamlit Cloud

---

## Streamlit Workflow

```text
User
   │
   ▼
Chat Interface
   │
   ▼
Finance Question
   │
   ▼
FAISS Vector Search
   │
   ▼
Relevant Financial Chunks
   │
   ▼
Llama 3.1 (Groq)
   │
   ▼
Answer
   │
   ▼
Display Sources
```

---

## Advantages

- Interactive web interface
- Accessible through any browser
- Easy deployment
- Recruiter-friendly project
- Portfolio-ready application